In [11]:
# imports
import pandas as pd
import tensorflow as tf
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from constant import DATA_DIR, RANDOM_STATE


# Viewing the Data w/ Pandas

In [12]:
# loading the data
clf_df = pd.read_csv(f'{DATA_DIR}/draft.csv')

In [13]:
# looking at what the data looks like
clf_df.head(),clf_df.dtypes

(  position  longsnapper  height  weight   draftage  bench  forty  \
 0       OL            0   81.25   322.0  22.112329     19   5.40   
 1       OL            0   81.00   317.0  24.227397     16   5.46   
 2       OL            0   80.75   324.0  22.520548     18   5.53   
 3       OL            0   80.00   335.0  23.276712     17   0.00   
 4       OL            0   80.00   320.0  25.243836     22   0.00   
 
    fortyelectric  verticaljump  broadjump  threecone  shuttle  drafted  
 0          5.430         28.50     95.000      7.470    4.660        1  
 1          5.790         26.75     96.624      8.280    5.000        1  
 2          5.540         24.50     87.500      8.180    4.900        1  
 3          5.453         28.00     96.000      8.047    4.918        1  
 4          5.280         33.50     93.000      7.500    4.610        1  ,
 position             str
 longsnapper        int64
 height           float64
 weight           float64
 draftage         float64
 bench   

In [14]:
# one hot encoding the position
def position_map_to_numerical(df):
    """A function that takes in a df. Grabs all the unique str within
    the `position` col. Then convert them into numerical values.

    It does not make sense to map each position label to a numerical value
    in acsending order because then we are saying one position might be better than
    the other. This is the differeniation between categorical and ordinal, as in,
    position is categorical, thus, we need a one hot encoding.
    see<https://towardsdatascience.com/encoding-categorical-variables-in-machine-learning-5644a43c123f/>
    Thus, we want to one hot encode it.
    """

    # TODO(shenukp1) Nov 15, 2026: might need a hash map of this later
    # unique_arr = df["position"].unique()
    
    # see<https://www.geeksforgeeks.org/machine-learning/one-hot-encoding-from-a-pandas-column-containing-a-list/>
    one_hot = pd.get_dummies(df['position'].apply(pd.Series).stack()).groupby(level=0).sum()
    df_one_hot = one_hot.join(df.drop('position', axis=1))
    return df_one_hot
clf_df = position_map_to_numerical(clf_df)
clf_df.head()
    

,DB,DL,K,LB,OL,QB,RB,WR,longsnapper,height,weight,draftage,bench,forty,fortyelectric,verticaljump,broadjump,threecone,shuttle,drafted
0,0,0,0,0,1,0,0,0,0,81.25,322.0,22.112329,19,5.40,5.430,28.50,95.000,7.470,4.660,1
1,0,0,0,0,1,0,0,0,0,81.00,317.0,24.227397,16,5.46,5.790,26.75,96.624,8.280,5.000,1
2,0,0,0,0,1,0,0,0,0,80.75,324.0,22.520548,18,5.53,5.540,24.50,87.500,8.180,4.900,1
3,0,0,0,0,1,0,0,0,0,80.00,335.0,23.276712,17,0.00,5.453,28.00,96.000,8.047,4.918,1
4,0,0,0,0,1,0,0,0,0,80.00,320.0,25.243836,22,0.00,5.280,33.50,93.000,7.500,4.610,1


In [15]:
# Now we need to split this into x(data) and prediction(y) variables
X, Y = clf_df.iloc[:,:-1],clf_df.iloc[:,-1]
X,Y

(     DB  DL  K  LB  OL  QB  RB  WR  longsnapper  height  weight   draftage  \
 0     0   0  0   0   1   0   0   0            0  81.250   322.0  22.112329   
 1     0   0  0   0   1   0   0   0            0  81.000   317.0  24.227397   
 2     0   0  0   0   1   0   0   0            0  80.750   324.0  22.520548   
 3     0   0  0   0   1   0   0   0            0  80.000   335.0  23.276712   
 4     0   0  0   0   1   0   0   0            0  80.000   320.0  25.243836   
 ..   ..  .. ..  ..  ..  ..  ..  ..          ...     ...     ...        ...   
 666   0   0  0   0   0   0   0   1            0  67.875   176.0  23.298630   
 667   1   0  0   0   0   0   0   0            0  67.625   176.0  24.380822   
 668   0   0  0   0   0   0   1   0            0  67.375   182.0  25.627397   
 669   0   0  1   0   0   0   0   0            0  67.000   165.0  21.753425   
 670   0   0  0   0   0   0   1   0            0  66.875   168.0  24.550685   
 
      bench  forty  fortyelectric  verticaljump  b

In [17]:
# The lectures mentioned that a 80/20 split is industry standard
X_train, X_test,y_train, y_test = train_test_split(X,Y,
                                   random_state=RANDOM_STATE, 
                                   test_size=0.20)


In [21]:
X_train.shape, X_test.shape,y_train.shape,y_test.shape

((536, 19), (135, 19), (536,), (135,))

In [22]:
# training time
model = tf.keras.Sequential([
 tf.keras.layers.Dense(512, activation='relu'),
 tf.keras.layers.Dense(1)
])
model.compile(optimizer='adam',
              loss='binary_crossentropy',# because we have a binary input
              metrics=['accuracy'])